# Dynamic Gradient Flow Monitoring

## Install

```bash
pip install graphyco
```

---

## Setup & Bridge Initialization

Initialize DynamicExecutionBridge on a residual block.

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from graphyco.bridge.grad_bridge import DynamicExecutionBridge


class ResNetBlock(nn.Module):
    """Simple residual block for demonstration."""
    def __init__(self, dim=64):
        super().__init__()
        self.l1 = nn.Linear(dim, dim)
        self.r1 = nn.ReLU()
        self.l2 = nn.Linear(dim, dim)
        self.r2 = nn.ReLU()
        self.out = nn.Linear(dim, 10)

    def forward(self, x):
        h = self.l2(self.r1(self.l1(x)))
        return self.out(self.r2(h + x))


torch.manual_seed(42)
model = ResNetBlock(dim=64)
bridge = DynamicExecutionBridge(model, mode="trace")
print(f"Traced {len(bridge.static_state.nodes)} nodes and {len(bridge.static_state.edges)} edges.")

Traced 8 nodes and 8 edges.


## Multi-Step Execution & Flow Tracking

Execute 5 forward/backward steps with loss computation.

In [2]:
inputs = torch.randn(8, 64)
opt = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(5):
    batch = inputs + 0.05 * torch.randn_like(inputs)
    state = bridge.step(inputs=batch, optimizer=opt, loss_fn=lambda out: out.sum())
    bneck = state.dynamic_evaluation["bottleneck"]
    print(f"Step {state.step}: Bottleneck = '{bneck['bottleneck_node']}', G_max = {bneck['max_concentration']:.3f}")

Step 1: Bottleneck = 'out', G_max = 2.774
Step 2: Bottleneck = 'out', G_max = 2.059
Step 3: Bottleneck = 'out', G_max = 1.366
Step 4: Bottleneck = 'x', G_max = 1.677
Step 5: Bottleneck = 'x', G_max = 4.059


## Node Records and Tensor Statistics

Inspect activation RMS, gradient RMS, and sensitivity ratio per node.

In [3]:
latest = bridge.trajectory[-1]
rows = []
for nid, rec in sorted(latest.node_records.items()):
    act_rms = rec.activation.rms if rec.activation else 0.0
    grad_rms = rec.activation_gradient.rms if rec.activation_gradient else 0.0
    rows.append({
        "Node": nid,
        "Type": rec.node_type,
        "Has Params": rec.has_parameters,
        "Act RMS": f"{act_rms:.4f}",
        "Grad RMS": f"{grad_rms:.4f}",
        "Ratio (Grad/Act)": f"{rec.ratio_grad_act:.4f}" if rec.ratio_grad_act is not None else "N/A"
    })
pd.DataFrame(rows)

,Node,Type,Has Params,Act RMS,Grad RMS,Ratio (Grad/Act)
0,add,call_function,False,8.0325,2.2623,0.2816
1,l1,call_module,True,2.0038,8.5132,4.2486
2,l2,call_module,True,7.7473,2.2623,0.2920
3,out,call_module,True,113.8632,1.0000,0.0088
4,output,output,False,113.8632,1.0000,0.0088
5,r1,call_module,False,1.9421,10.2974,5.3021
6,r2,call_module,False,8.0069,2.2845,0.2853
7,x,placeholder,False,1.0129,28.4405,28.0773


## Directed Edge Gradient Attenuation

Inspect backward attenuation ratio beta and logarithmic attenuation Delta along edges.

In [4]:
attenuation = latest.dynamic_evaluation["attenuation"]
edge_rows = []
for edge_key, att in list(attenuation.items())[:6]:
    edge_rows.append({
        "Edge": edge_key,
        "Backward Ratio": f"{att['backward_ratio']:.4f}",
        "Log Ratio": f"{att['log_backward_ratio']:+.4f}",
        "Src Grad L2": f"{att['source_grad_l2']:.4f}",
        "Tgt Grad L2": f"{att['target_grad_l2']:.4f}"
    })
pd.DataFrame(edge_rows)

,Edge,Backward Ratio,Log Ratio,Src Grad L2,Tgt Grad L2
0,x->l1,0.2993,-1.2062,643.5354,192.6313
1,l1->r1,1.2096,+0.1903,192.6313,233.0034
2,r1->l2,0.2197,-1.5155,233.0034,51.1910
3,l2->add,1.0000,+0.0000,51.1910,51.1910
4,x->add,0.0795,-2.5314,643.5354,51.1910
5,add->r2,1.0098,+0.0097,51.1910,51.6914


## Structural-Functional Classification (Types I - IV)

Categorize nodes into 4 quadrants based on topology vs. gradient magnitude.

In [5]:
corr = latest.dynamic_evaluation["correspondence"]
print("Node Category Counts:")
for cat, count in corr["counts"].items():
    print(f"  {cat:<10}: {count}")

print()
print(f"Spearman rho(Topology, Gradient): {corr['spearman_topo_grad']:.4f} (p={corr['spearman_pvalue']:.4e})")

Node Category Counts:
  Type I    : 4
  Type II   : 3
  Type III  : 0
  Type IV   : 1

Spearman rho(Topology, Gradient): 0.2209 (p=5.9915e-01)


## Temporal Stability Analysis

Compute temporal coefficient of variation (CV = sigma / mu) across execution steps.

In [6]:
temporal = latest.dynamic_evaluation.get("temporal", {})
t_rows = []
for nid, t_stat in sorted(temporal.items())[:6]:
    t_rows.append({
        "Node": nid,
        "Mean Grad": f"{t_stat['mean_gradient']:.4f}",
        "Std Grad": f"{t_stat['std_gradient']:.4f}",
        "CV": f"{t_stat['cv']:.4f}",
        "Classification": t_stat["classification"]
    })
pd.DataFrame(t_rows)

,Node,Mean Grad,Std Grad,CV,Classification
0,add,0.9280,0.7507,0.8089,unstable
1,l1,2.1227,3.2357,1.5243,unstable
2,l2,0.9280,0.7507,0.8089,unstable
3,out,1.0000,0.0000,0.0000,consistently_weak
4,output,1.0000,0.0000,0.0000,consistently_weak
5,r1,2.5833,3.9059,1.5120,unstable


## Serialization to JSON and CSV

Export monitored trajectory to disk.

In [7]:
json_path = "resnet_step5.json"
csv_path = "resnet_step5_nodes.csv"
latest.to_json(path=json_path)
latest.to_csv_nodes(path=csv_path)

print(f"Exported JSON size: {os.path.getsize(json_path):,} bytes")
print(f"Exported CSV size : {os.path.getsize(csv_path):,} bytes")

# Clean up exported files
if os.path.exists(json_path): os.remove(json_path)
if os.path.exists(csv_path): os.remove(csv_path)

Exported JSON size: 27,471 bytes
Exported CSV size : 1,090 bytes
